# Study Buddy V10 — Free GPU Tutor Prototype

This notebook is the first GPU milestone. It checks the temporary NVIDIA GPU, installs the open-source LiveTalking engine, and prepares the Wav2Lip avatar path. LiveTalking supports WebRTC, interruption, custom avatars, and Wav2Lip/MuseTalk.

**Important:** Colab GPU sessions are temporary. This notebook is for proving the model pipeline; it is not permanent hosting.

In [ ]:
!nvidia-smi
import sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 1. Clone LiveTalking

The current LiveTalking project documents Python 3.12 + PyTorch 2.9.1/CUDA 12.8 for its latest setup. If the Colab runtime already has a compatible recent PyTorch, we avoid unnecessarily replacing it.

In [ ]:
%cd /content
!rm -rf LiveTalking
!git clone --depth 1 https://github.com/lipku/LiveTalking.git
%cd /content/LiveTalking
!pip -q install -r requirements.txt
print('LiveTalking source and Python dependencies installed.')

## 2. Download the Wav2Lip test assets

LiveTalking's documented quick start uses `wav2lip256.pth` plus the `wav2lip256_avatar1` avatar archive. The project publishes the model files through its Google Drive model folder.

If the Drive folder blocks automated download, download the two files manually and upload them into `/content/LiveTalking/models` and `/content/LiveTalking/data/avatars`.

In [ ]:
!pip -q install gdown
!mkdir -p /content/livetalking_downloads
!gdown --folder 'https://drive.google.com/drive/folders/1FOC_MD6wdogyyX_7V1d4NDIO7P9NlSAJ?usp=sharing' -O /content/livetalking_downloads || true
!find /content/livetalking_downloads -maxdepth 3 -type f -printf '%p\n' | head -100

## 3. Put the Wav2Lip files in the expected locations

Run the next cell. If it finds the files, it copies them automatically. If not, use the printed instructions.

In [ ]:
import os, glob, shutil, tarfile
root='/content/LiveTalking'
download='/content/livetalking_downloads'
os.makedirs(f'{root}/models', exist_ok=True)
os.makedirs(f'{root}/data/avatars', exist_ok=True)
pths=glob.glob(download+'/**/wav2lip256.pth', recursive=True)
if pths:
    shutil.copy2(pths[0], f'{root}/models/wav2lip.pth')
    print('Copied wav2lip256.pth -> models/wav2lip.pth')
else:
    print('wav2lip256.pth was not found. Upload it to /content/LiveTalking/models/wav2lip.pth')
archives=glob.glob(download+'/**/wav2lip256_avatar1.tar.gz', recursive=True)
if archives:
    with tarfile.open(archives[0], 'r:gz') as t:
        t.extractall(f'{root}/data/avatars')
    print('Extracted wav2lip256_avatar1 avatar archive.')
else:
    print('wav2lip256_avatar1.tar.gz was not found. Upload/extract it into /content/LiveTalking/data/avatars/')
print('Model files present:', os.path.exists(f'{root}/models/wav2lip.pth'))
print('Avatar directories:', os.listdir(f'{root}/data/avatars')[:20])

## 4. Start the digital-human server

LiveTalking's documented command is below. A normal public browser connection requires TCP 8010 plus a broad UDP range for WebRTC, so Colab is **not** our final hosting solution. The purpose here is to prove the GPU inference engine starts successfully.

In [ ]:
%cd /content/LiveTalking
!python app.py --transport webrtc --model wav2lip --avatar_id wav2lip256_avatar1 --max_session 1

## Next milestone

After the server starts successfully, we move to **V10.1**: browser microphone -> interruption -> Groq response -> TTS -> LiveTalking WebRTC. Then we replace Wav2Lip with MuseTalk 1.5 for the higher-quality avatar.